In [ ]:
#==================================
# DEM提取脚本 - 支持断点重传,进度监控
# 作者: King
# 日期: 2026-02-21
# 描述: 从国家地图服务提取DEM数据，针对每个样点进行提取，支持断点重传，实时显示进度和速度。
#==================================


import arcpy
import os
import time

# ==================== 参数设置 ====================
service = "https://elevation.nationalmap.gov/arcgis/services/3DEPElevation/ImageServer"
buffer_layer = r"C:\Users\King\Documents\ArcGIS\Projects\MyProject8\MyProject8.gdb\GenerateTessellation_1_ExportFeatures2"
out_folder = r"C:\Users\King\Desktop\大三下学习生活\prj-Alban\data\raw\DEM\DEM_Evelation_Fishnet"

if not os.path.exists(out_folder):
    os.makedirs(out_folder)

# 强制像元大小为1米
arcpy.env.cellSize = 1

# ==================== 初始化监控 ====================
# 获取总点数用于计算进度
#arcpy.env.overwriteOutput = True
arcpy.env.extent = None  # 确保全局范围清空

#
total_points = int(arcpy.management.GetCount(buffer_layer)[0])

print(f"--- 任务启动：共计 {total_points} 个样点 ---")

start_time = time.time()
success_count = 0
skip_count = 0
error_count = 0

# 这一步确保后续请求的是 Raw Elevation 原始数据
dem_layer_name = "Raw_Elevation_Layer"
arcpy.management.MakeImageServerLayer(service, dem_layer_name, processing_template="None") #这里的template参数根据实际情况设置，如果不需要可以保持为"None"， 字段为ArcGIS Pro 中 图层属性-处理模板，若无请留空
print("正在连接服务器并应用 '-None' 模板...")


# ==================== 核心循环 ====================
with arcpy.da.SearchCursor(buffer_layer, ["SHAPE@", "OID@"]) as cursor:
    for row in cursor:
        feat = row[0]
        oid = row[1]
        
        out_name = f"Extract_DEM_{oid}.tif"
        out_path = os.path.join(out_folder, out_name)
        
        # --- 1. 断点重传逻辑 ---
        # 如果文件已经存在，直接跳过，进入下一个
        if os.path.exists(out_path):
            skip_count += 1
            continue

        # --- 2. 设置分析范围，取消选择 ---
        arcpy.env.extent = feat.extent
        #print(arcpy.env.extent)
        arcpy.env.extent = None  # 确保全局范围清空

        try:
            # --- 3. 执行提取 ---

            arcpy.sa.ExtractByMask(dem_layer_name, feat).save(out_path)
            success_count += 1
            
            # --- 4. 速度与时间可视化 ---
            current_processed = success_count + skip_count
            elapsed_time = time.time() - start_time
            # 计算平均速度 (个/秒)
            speed = current_processed / elapsed_time if elapsed_time > 0 else 0
            # 计算剩余时间 (分钟)
            remaining_points = total_points - current_processed
            eta_min = (remaining_points / speed / 60)
            eta_hours = eta_min / 60
            # 格式化输出进度
            status_msg = (f"进度: {current_processed}/{total_points} | "
                          f"成功: {success_count} | 跳过: {skip_count} | "
                          f"速度: {speed:.2f} 点/秒 | "
                          f"预计剩余: {eta_min:.1f} 分钟"
                          f"，{eta_hours:.1f} 小时")
            
            # 每完成 1 个点更新一次，或根据需要改为每 5 个点更新
            print(status_msg)
            # 同时将消息发送到 ArcGIS Pro 的地理处理历史记录窗口
            arcpy.AddMessage(status_msg)

        except Exception as e:
            error_count += 1
            error_msg = f"!!! 点 {oid} 失败: {e}"
            print(error_msg)
            arcpy.AddWarning(error_msg)

# 重置环境
arcpy.ClearEnvironment("extent")

# ==================== 最终报告 ====================
total_time_min = (time.time() - start_time) / 60
print(f"\n--- 提取任务结束 ---")
print(f"总计完成: {success_count}")
print(f"断点跳过: {skip_count}")
print(f"失败记录: {error_count}")
print(f"总耗时: {total_time_min:.2f} 分钟")

--- 任务启动：共计 415 个样点 ---
正在连接服务器并应用 '-None' 模板...
进度: 1/415 | 成功: 1 | 跳过: 0 | 速度: 0.01 点/秒 | 预计剩余: 965.8 分钟
进度: 2/415 | 成功: 2 | 跳过: 0 | 速度: 0.01 点/秒 | 预计剩余: 928.2 分钟
进度: 3/415 | 成功: 3 | 跳过: 0 | 速度: 0.01 点/秒 | 预计剩余: 941.9 分钟
